# Drinking Water Demand Forecasting with Recurrent Neural Networks - Part 1
## One-step ahead prediction with Recurrent Neural Networks

This series of Jupyter Notebook exercises focuses on implementing Recurrent Neural Networks (RNN) in `PyTorch`. Throughout this series, you will gain a better understanding of RNNs, a class of neural networks particularly adept at processing sequential data. We will learn by approaching a time series forecasting in the water domain. Of course, the skills and knowledge you acquire here are not limited to this area.

Please go through the Problem Definition and Walkthrough sections of this notebooks, until you reach the Assignments.

## Problem Definition

Water Distribution Systems (WDS) are networked infrastructures designed to provide drinking water to citizens and businesses. Like other critical infrastructure, modern WDS rely on digitalisation to optimize their operations in order to maximise efficiency by ensuring sustainable water management practices.

Digitalisation allows gathering and analysing sensor data to inform control strategies. For instance, accurate forecasting of _drinking water demands_ over shor-time horizons (e.g., 24 hours) can be used to better manage the operations of the pumping stations of the WDS, as well as optimizing the production of drinking water from water treatment plants.

Water demand time series are usually recorded by flow meters located at the inlet of a district. This data can then be used to develop forecasting methods, which are usually based on time series analysis techniques or, more recently, machine learning (ML). Typical input data for these models are past water demands, meteorological data, and calendar information (e.g., day of the week, holidays, important events, ...).


<br/>


<center><figure>
  <img src="https://ars.els-cdn.com/content/image/1-s2.0-S0377042716300565-gr6_lrg.jpg" width=600/>
<figcaption>Figure 1. Water demand forecasting: observed data vs machine learning prediction.
    
<sub><sup>Image credits: Brentan et al. (2017), https://doi.org/10.1016/j.wroa.2019.100028 </sup></sub></figcaption>
</figure></center>

The data for this assignment was kindly provided by the authors of [1]. The data refers to hourly measurments from a real WDS serving around 120k users (residential and industrial) of a medium-sized city of Northern Italy. The data spans two years and includes hourly average water demand in [L/s] (liters per second). You will be using the data to _train_ and _validate_ different Deep Learning (DL) models based on Feed Forward Neural Networks and Recurrent Neural Networks.

In this first notebook, the goal is to compare different architectures for the **one-step ahead forecasting of water demand**.

<u> References </u>

[1] Pacchin, E., Gagliardi, F., Alvisi, S. and Franchini, M., 2019. A comparison of short-term water demand forecasting models. Water resources management, 33(4), pp.1481-1497.


## Load python modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import time
import os
from urllib.request import urlretrieve

from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# Check if CUDA is available, otherwise use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Download and load the data

In [ ]:
# Download the water demand dataset
year_1 = "https://surfdrive.surf.nl/files/index.php/s/T2SN8dnXXYWz4q5/download"
year_2 = "https://surfdrive.surf.nl/files/index.php/s/StRwCboQoDPRGdD/download"

data_folder = "data"
water_demand_file1 = os.path.join(data_folder, "water_demand_Y1.txt")
water_demand_file2 = os.path.join(data_folder, "water_demand_Y2.txt")

if not os.path.isfile(water_demand_file1):
    print("Downloading dataset...")
    os.makedirs("data", exist_ok=True)
    urlretrieve(year_1, water_demand_file1)
    urlretrieve(year_2, water_demand_file2)

## Exploratory data analysis

We are loading data from the two years and concatenating them for joint analysis.

> Note: You can choose to work with only one of the two years to expedite training.


In [ ]:
# load water demand data
df_year1 = pd.read_csv(water_demand_file1,header=None, sep=r"\s+")
df_year2 = pd.read_csv(water_demand_file2,header=None, sep=r"\s+")
df = pd.concat([df_year1, df_year2], axis = 0).reset_index(drop=True)
df.head()

Let's visualize the typical pattern of a random day in our data. In general, these patterns show two peaks of consumption: in the morning and in the evening. Consumption is at its lowest during late night and early mornings.

In [ ]:
# plot random day
day_ix = np.random.randint(len(df))
ax = df.loc[day_ix].plot()
ax.set_xlabel('Hour of the  day');
ax.set_ylabel('Average hourly demand [L/s]');
ax.set_title(f'Water consumption for day #{day_ix}');

We can obtain more information on the hour-by-hour distribution of the water demand by using boxplots. These are computed across all days for each hour of the day. By plotting the distribution for both years we see that <u> consumptions increased in Year #2 </u>. This also emerges from the overall average (red line).

In [ ]:
# plot hourly water demand distributions across all days

# year 1
f, axes = plt.subplots(1,2,figsize=(20,6))
df_year1.boxplot(ax=axes[0])
axes[0].set_xlabel('Hour of the  day');
axes[0].set_ylabel('Average hourly demand [L/s]');
axes[0].set_title('Hour-by-hour distribution of water demand for Year#1');
axes[0].set_ylim([350,1450])

avg_year1 = df_year1.values.reshape(-1).mean()
axes[0].hlines(avg_year1,1,24, colors='r')
axes[0].text(2, 1000, f'avg={avg_year1:.2f}', fontsize=12)

# year 2
df_year2.boxplot(ax=axes[1])
axes[1].set_xlabel('Hour of the  day');
# axes[1].set_ylabel('Average hourly demand [L/s]');
axes[1].set_title('Hour-by-hour distribution of water demand for Year#2');
axes[1].set_ylim([350,1450])

avg_year2 = df_year2.values.reshape(-1).mean()
axes[1].hlines(avg_year2,1,24, colors='r')
axes[1].text(2, 1000, f'avg={avg_year2:.2f}', fontsize=12)

f.tight_layout()

## Creation of dataset and data normalization/standardization

The following code snippets serve two essential functions:

1. `create_sequences`:
   - This function creates input/output sequences from a given time series.
   - The input sequence spans T time steps, from time t to time t+T (excluded).
   - The output sequence spans H time steps, from time t+T to time t+T+H (excluded).
   - In the current notebook, we use H = 1 for predicting one step ahead. And T = 168, that is all hourly data up to 1 week before.

2. `scale_sequences`:
   - This function scales or normalizes sequences.
   - If a scaler is provided, it transforms the sequences using that scaler.
   - If no scaler is provided, it creates a scaler based on the specified `scaler_type` ('standard' or 'minmax') and then transforms the sequences.
   - The scaled sequences are returned along with the scaler used (or `None` if a scaler was provided).


In [ ]:
def create_sequences(series,T=168,H=24):
    # This function creates a dataset of input/output sequences from a time series.
    # The input sequence is T steps long, from time t to time t+T (excluded).
    # The output sequence is H steps long, from time t+T to time t+T+H (excluded).
    X = []
    Y = []
    for t in range(len(series)-T-H):
        x = series[t:t+T]
        X.append(x)
        y = series[t+T:t+T+H]
        Y.append(y)
    X = np.array(X)
    Y = np.array(Y)
    return X,Y

def scale_sequences(X,scaler=None,scaler_type='standard'):
    # Uses a standard scaler to transform sequences. The scaler is created if no scaler is passed as argument.
    Xshape=X.shape
    if scaler:
        X = scaler.transform(X.reshape(-1,1)).reshape(Xshape)
        return X
    else:
        if scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'minmax':
            scaler = MinMaxScaler()
        else:
            raise Exception("Type of scikit-learn scaler not supported. Choose 'standard' or 'minmax.")
        X = scaler.fit_transform(X.reshape(-1,1)).reshape(Xshape)
        return X, scaler

We now create the sequence datasets for _training_, _validating_ and _testing_ our DL methods.

In [ ]:
T = 168     # number of time steps to use for prediction (168 hours = 1 week)
H = 1       # number of time steps to predict (1 hour)
X, Y = create_sequences(df.values.reshape(-1),T=T,H=H)
print(X.shape)
print(Y.shape)

As you can see, each input-ouput pair is made of: 1) an input sequence of 168 hours (e.g., 1 week) and 2) an output sequence of 1 hours.

In [ ]:
random_ix = np.random.choice(X.shape[0])
f, ax = plt.subplots(1,figsize=(10,3))
ax.plot(np.arange(T),X[0], label='X (input)')
ax.plot(np.arange(T,T+H),Y[0], 'x', label = 'Y (output)')
ax.set_title(f'X and Y for sequence #{random_ix} of the Year #1 dataset');
ax.legend();

These sequences are not normalized yet, and we need to split them into training and validation. We first perform the latter task and then we use ```scale_sequences``` to perform the scaling. Is important to use the scaler "fitted" for the training dataset to scale tha validation dataset and then the test dataset). Since the input and output variables are both water demand, we only need to fit one scaler. ```scale_sequences``` recognize whether to fit a scaler or use an existing one based on the arguments it receives when called.

In [ ]:
# We keep track of indexes of train and validation.
X_tra, X_tst, Y_tra, Y_tst, ix_tra, ix_tst = train_test_split(
    X, Y, np.arange(X.shape[0]), test_size=0.30, shuffle=True, random_state=42)

# Split the existing test dataset into validation and test sets (50/50 split)
X_val, X_tst, Y_val, Y_tst, ix_val, ix_tst = train_test_split(
    X_tst, Y_tst, ix_tst, test_size=0.5, shuffle=True, random_state=42)


print(f"X_tra.shape: {X_tra.shape}, Y_tra.shape: {Y_tra.shape}")
print(f"X_val.shape: {X_val.shape}, Y_val.shape: {Y_val.shape}")
print(f"X_tst.shape: {X_tst.shape}, Y_tst.shape: {Y_tst.shape}")

Let's plot a random sequence from the training dataset

In [ ]:
random_ix = np.random.choice(X_tra.shape[0])
f, ax = plt.subplots(1,figsize=(10,3))
ax.plot(np.arange(T),X_tra[random_ix], label='Xtra')
ax.plot(np.arange(T,T+H),Y_tra[random_ix], 'x', label = 'Ytra')
ax.set_title(f'X and Y for sequence #{random_ix} of the training dataset');

Now we scale all data. Remember: <u>you "fit" the scaler using training data </u>; when you develop your machine learning models, you have to assume that you have no information on validation and test data.

In [ ]:
# scale/normalize
X_tra, scaler = scale_sequences(X_tra, scaler_type='standard')
Y_tra = scale_sequences(Y_tra, scaler)
X_val = scale_sequences(X_val, scaler)
Y_val = scale_sequences(Y_val, scaler)
X_tst = scale_sequences(X_tst, scaler)
Y_tst = scale_sequences(Y_tst, scaler)

As you can see, the sequences before and after normalization look the same, but the ranges are different.

In [ ]:
f, ax = plt.subplots(1,figsize=(10,3))
ax.plot(np.arange(T),X_tra[random_ix], label='Xtra')
ax.plot(np.arange(T,T+H),Y_tra[random_ix], 'x', label = 'Y (output)')
ax.set_title(f'X and Y for sequence #{random_ix} of the SCALED training dataset');

## Definition of MultiLayer Perceptron and Simple Recurrent Neural Network

Now, we will compare the performance of a Multilayer Perceptron (MLP) against a Simple Recurrent Neural Network (RNN). This comparison aims to assess whether the RNN's sequential inductive bias provides any advantages in modeling and predicting our time series data.


In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(MLP, self).__init__()
        # Define the layers of the network
        self.fc1 = nn.Linear(T, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Define the forward pass
        x = F.tanh(self.fc1(x))  # Activation function (ReLU) after first layer
        x = F.tanh(self.fc2(x))  # Activation function (ReLU) after second layer
        x = self.fc3(x)          # Output layer
        return x

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        # RNN layer
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden_size, batch_first=True)

        # Output layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Reshape input to have feature dimension of 1
        x = x.unsqueeze(-1)   # Assuming input x has shape (batch, sequence)

        # RNN layer
        x, hn = self.rnn(x)   # We do not need the hidden states hn

        # Select the output of the last time step
        x = x[:, -1, :]

        # Output layer
        x = self.fc(x)

        return x

### Functions for training loop end evaluation

In [ ]:
def evaluate_model(model, test_loader, criterion, device):
    model.eval()  # Set the model to evaluation mode
    test_loss = 0

    with torch.no_grad():  # No need to track gradients during evaluation
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            test_loss += loss.item()

    avg_test_loss = test_loss / len(test_loader)
    return avg_test_loss

def train_and_validate(model, train_loader, val_loader, criterion, optimizer, num_epochs, device, save_path):
    best_val_loss = float("inf")  # Track the best validation loss
    train_losses = []
    val_losses = []

    start_time = time.time()  # Start training time

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        total_train_loss = 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation Phase
        avg_val_loss = evaluate_model(model, val_loader, criterion, device)
        val_losses.append(avg_val_loss)

        # Save Best Model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)

        if (epoch + 1) % 20 == 0:
            print(f'Epoch {epoch+1}/{num_epochs}', f'Train Loss: {avg_train_loss:.4f}, '
                  f'Validation Loss: {avg_val_loss:.4f}', f'Best Validation Loss: {best_val_loss:.4f}')
    train_time = time.time() - start_time
    print("Training complete.")
    return train_losses, val_losses, best_val_loss, train_time

### MLP vs SimpleRNN comparison


We create the `TensorDatasets` and the `DataLoaders`. We also set equal training epochs and learning rate for both architectures.

In [ ]:
train_dataset = TensorDataset(torch.tensor(X_tra, dtype=torch.float32), torch.tensor(Y_tra, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(Y_val, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_tst, dtype=torch.float32), torch.tensor(Y_tst, dtype=torch.float32 ))

In [ ]:
batch_size = 256      # You can modify this based on your requirements

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
num_epochs = 200
lr = 0.0005

#### Training MLP

In [ ]:
model_MLP = MLP(256, H).to(device)

In [ ]:
# Loss function
criterion = nn.MSELoss()  # Or another appropriate loss function

optimizer = optim.AdamW(model_MLP.parameters(), lr=lr)
save_path_MLP = './models/MLP_model.pth'

if not os.path.exists('./models'):
    os.makedirs('./models')

In [ ]:
train_losses_MLP, val_losses_MLP, best_val_loss_MLP, time_MLP = train_and_validate(model_MLP, train_loader, val_loader, criterion, optimizer, num_epochs, device, save_path_MLP)

In [ ]:
plt.plot(train_losses_MLP, label='Training Loss')
plt.plot(val_losses_MLP, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Losses - MLP')
plt.legend()
plt.show()

In [ ]:
# Load the best model
model_MLP.load_state_dict(torch.load(save_path_MLP))

#### Training SimpleRNN

In [ ]:
model_RNN = SimpleRNN(128, H).to(device)

In [ ]:
# Loss function
criterion = nn.MSELoss()  # Or another appropriate loss function

optimizer = optim.AdamW(model_RNN.parameters(), lr=lr)
save_path_RNN = './models/RNN_model.pth'

In [ ]:
train_losses_RNN, val_losses_RNN, best_val_loss_RNN, time_RNN = train_and_validate(model_RNN, train_loader, val_loader, criterion, optimizer, num_epochs, device, save_path_RNN)

In [ ]:
# Load the best model
model_RNN.load_state_dict(torch.load(save_path_RNN))

In [ ]:
plt.plot(train_losses_RNN, label='Training Loss')
plt.plot(val_losses_RNN, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Losses - RNN')
plt.legend()
plt.show()

### Comparison

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

avg_test_loss_MLP = evaluate_model(model_MLP, test_loader, criterion, device)
avg_test_loss_RNN = evaluate_model(model_RNN, test_loader, criterion, device)

print(f"MLP --> num. trainable parameters:{count_parameters(model_MLP):8d} | Best Val Loss: {best_val_loss_MLP:.4f} | Test loss: {avg_test_loss_MLP:.4f} | Training time: {time_MLP:.2f}")
print(f"RNN --> num. trainable parameters:{count_parameters(model_RNN):8d} | Best Val Loss: {best_val_loss_RNN:.4f} | Test loss: {avg_test_loss_RNN:.4f} | Training time: {time_RNN:.2f}")

In [ ]:
model_MLP.eval()
model_RNN.eval()

f,axes = plt.subplots(3,3, figsize=(20,10))

for ix, ax in enumerate(axes.reshape(-1)):
  with torch.no_grad():
      # Randomly select a sample from the test dataset
      random_index = random.randint(0, len(test_dataset) - 1)
      inputs, target = test_dataset[random_index]
      inputs, target = inputs.to(device).unsqueeze(0), target.to(device).unsqueeze(0)

      # Predict
      prediction_MLP = model_MLP(inputs)
      prediction_RNN = model_RNN(inputs)

      # Rescale inputs, target, and prediction using the scaler
      inputs_rescaled = scaler.inverse_transform(inputs.cpu().numpy().flatten().reshape(-1, 1)).flatten()
      target_rescaled = scaler.inverse_transform(target.cpu().numpy().flatten().reshape(-1, 1)).flatten()
      prediction_MLP_rescaled = scaler.inverse_transform(prediction_MLP.cpu().numpy().flatten().reshape(-1, 1)).flatten()
      prediction_RNN_rescaled = scaler.inverse_transform(prediction_RNN.cpu().numpy().flatten().reshape(-1, 1)).flatten()

      # Plotting inputs (time series)
      ax.plot(np.arange(T), inputs_rescaled, label='Inputs')

      # Plotting prediction vs target
      ax.scatter(np.arange(T, T+H), target_rescaled, marker='x', label='Actual')
      ax.scatter(np.arange(T, T+H), prediction_MLP_rescaled, marker='o', facecolors='none', edgecolors='r', label='Predicted MLP')
      ax.scatter(np.arange(T, T+H), prediction_RNN_rescaled, marker='o', facecolors='none', edgecolors='g', label='Predicted RNN')

      ax.set_title(f'Test Example #{random_index}')
      ax.set_xlabel('Time Steps')
      ax.set_ylabel('Value')
      if ix == 0:
        ax.legend()

f.tight_layout()

## Assignments

1. **Store your best RNN and MLP**
   - We will need your best models for Part 2 of this series of notebooks. Make sure to save somewhere the files where you stored your best RNN and MLP models.
   - If you are using Google Colab, you can store these files after connecting your Google Drive (see previous notebooks). 
   - You can also quickly download them as follows:
 
     `from google.colab import files`
     
     `files.download(<folder of the file>/<file name>)`

2. **Comparison between RNN and MLP**
  - Critically analyze the performance differences between the Recurrent Neural Network (RNN) and the Multi-Layer Perceptron (MLP) models. Determine which model yielded better results and discuss the reasons for this performance difference.

3. **Loss Curve Analysis**
  - Review the training and validation loss curves for both the RNN and MLP models. Identify any notable patterns or anomalies in these curves. Describe these observations.

4. **[OPTIONAL] Tweak hyperparameters**
  - Play around with the complexity (e.g., size) of these networks and other hyperparameters such as the learning rate or number of epochs. Do you see any major changes in performance? If so, why? 

5. **[OPTIONAL] Change the length of the input sequence**
  - Decrease the length of the input sequence to one day and two days. Re-do the experiments for both MLP and RNN. You can work with your fellow classmates to proceed in parallel.
  - What do you see? Are the performance increasing or decreasing, or are they essentially the same? Critically analyze these results.